# L'importance des modèles

## Données

Repartons des données que nous avons mis en forme, et amenons progressivement une analyse plus avancée

In [1]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/pyshs/CUSO2026/refs/heads/main/data/css_openalex_26022026.csv")
df.head(2)

,id,type,primary_location,title,abstract_inverted_index,publication_year,publication_date,open_access,relevance_score,abstract,journal
0,https://openalex.org/W2159397589,article,"{'id': 'doi:10.1126/science.1167742', 'is_oa':...",Computational Social Science,"{'A': [0], 'field': [1], 'is': [2], 'emerging'...",2009.0,2009-02-06,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",1360.35770,A field is emerging that leverages the capacit...,Science
1,https://openalex.org/W2070907364,article,"{'id': 'doi:10.1140/epjst/e2012-01697-8', 'is_...",Manifesto of computational social science,NaN,2012.0,2012-11-01,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",497.82666,NaN,The European Physical Journal Special Topics


## Spacy pour aller vers les modèles

Installer Spacy et un modèle adapté

In [4]:
# pip install -U spacy

In [2]:
!python -m spacy download en_core_web_md

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 9.2 MB/s  0:00:03 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')


In [4]:
import spacy
pipeline = spacy.load("en_core_web_md")
text = "The language Python rocks, but the day is long."
doc = pipeline(text)

In [5]:
type(text)

str

In [6]:
type(doc)

spacy.tokens.doc.Doc

Le texte est "structuré" : composé de différents éléments qui n'existaient pas avant

POS, lemmatisation, etc.

In [12]:
doc

The language Python rocks, but the day is long.

In [13]:
for token in doc:
    print(token.text, token.lemma_, token.pos_, token.dep_, token.lemma_, token.is_oov)

The the DET det the False
language language NOUN compound language False
Python Python PROPN compound Python False
rocks rock NOUN ROOT rock False
, , PUNCT punct , False
but but CCONJ cc but False
the the DET det the False
day day NOUN nsubj day False
is be AUX conj be False
long long ADJ acomp long False
. . PUNCT punct . False


Récupérer uniquement les verbes ?

NER = Name Entity Recognition

In [14]:
text

'The language Python rocks, but the day is long.'

In [15]:
for ent in doc.ents:
    print(ent.text, ent.label_)

the day DATE


In [17]:
text2 = "We are March 20th"
pipeline(text2).ents

(March 20th,)

In [27]:
result = df["abstract"][0:10].dropna().apply(pipeline)

In [28]:
for r in result:
    print(r.ents)

()
()
()
(zero, zero, zero, 13, 25, English, today, CSS, two, 1, zero, 2)
(social geographic, GIS, Galileo, 2010, John Wiley &, Sons, Inc., Types and Structure &gt, Social Networks)
(first, ABM, ABM, second, Computational Social Science, CSS, ABM, CSS)


In [33]:
from spacy import displacy
displacy.render(result.iloc[3], style="ent", 
                jupyter=True)

Plongements

In [34]:
# Compare two documents
doc1 = pipeline(df["abstract"][0])
doc2 = pipeline(df["abstract"][1000])
print(doc1.similarity(doc2))

0.9585402950894589


## Utiliser des modèles d'HuggingFace : le cas de GliNER

Aller vers les modèles :
- HuggingFace
- Cartes des modèles
- Transformers

Un modèle plus complexe pour l'identification d'entités

Par exemple : https://huggingface.co/urchade/gliner_multi_pii-v1

In [2]:
#!pip install gliner
from gliner import GLiNER

model = GLiNER.from_pretrained("gliner-community/gliner_large-v2.5")

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

In [5]:
text = "Il se trouve qu'Émilien aime faire du Python. Et Léo aussi aime du Python"

labels = ["person", "programmation"]

entities = model.predict_entities(text, labels)

entities

[{'start': 16,
  'end': 23,
  'text': 'Émilien',
  'label': 'person',
  'score': 0.986863374710083},
 {'start': 38,
  'end': 44,
  'text': 'Python',
  'label': 'programmation',
  'score': 0.992074728012085},
 {'start': 49,
  'end': 52,
  'text': 'Léo',
  'label': 'person',
  'score': 0.9841655492782593},
 {'start': 67,
  'end': 73,
  'text': 'Python',
  'label': 'programmation',
  'score': 0.9955724477767944}]

In [6]:
df["abstract"][0:10].dropna().apply(lambda x:  model.predict_entities(x, ["digital method"]))

0                                                   []
2                                                   []
3    [{'start': 113, 'end': 141, 'text': 'computati...
5    [{'start': 9, 'end': 30, 'text': 'Large langua...
7    [{'start': 457, 'end': 497, 'text': 'automated...
9    [{'start': 45, 'end': 65, 'text': 'agent-based...
Name: abstract, dtype: object

## Application

Faire de l'analyse de sentiments

Une question : **quelles sont les prises de paroles les plus négatives ?**

- Embarras du choix
    - Par ex : [🚀 distilbert-based Multilingual Sentiment Classification Model
](https://huggingface.co/tabularisai/multilingual-sentiment-analysis)
- Comprendre le modèle / ce qu'il fait
- Importance d'évaluer son résultat